In [2]:
# Google Drive'ı bağladık
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Gerekli kütüphaneleri import et
import pandas as pd
import numpy as np

# Dosya yolu
file_path = '/content/drive/MyDrive/case_study_2/data/elektrik_veri_hashed.xlsx'

xls = pd.ExcelFile(file_path)
print("Sayfa isimleri:", xls.sheet_names)

Sayfa isimleri: ['Tahsilat', 'Tahsilat 1', 'Tahakkuk', 'Tahakkuk 1', 'Tahakkuk 2']


In [4]:
# Her sayfayı ayrı DataFrame'e yükle
df_tahsilat = pd.read_excel(xls, sheet_name='Tahsilat')
df_tahsilat_1 = pd.read_excel(xls, sheet_name='Tahsilat 1')
df_tahakkuk = pd.read_excel(xls, sheet_name='Tahakkuk')      # Hamamözü
df_tahakkuk_1 = pd.read_excel(xls, sheet_name='Tahakkuk 1')  # Gümüşhacıköy
df_tahakkuk_2 = pd.read_excel(xls, sheet_name='Tahakkuk 2')  # Göynücek

# DataFrame boyutlarını kontrol et
print(f"Tahsilat: {df_tahsilat.shape}")
print(f"Tahsilat 1: {df_tahsilat_1.shape}")
print(f"Tahakkuk (Hamamözü): {df_tahakkuk.shape}")
print(f"Tahakkuk 1 (Gümüşhacıköy): {df_tahakkuk_1.shape}")
print(f"Tahakkuk 2 (Göynücek): {df_tahakkuk_2.shape}")

Tahsilat: (636993, 9)
Tahsilat 1: (917632, 22)
Tahakkuk (Hamamözü): (124818, 10)
Tahakkuk 1 (Gümüşhacıköy): (765657, 10)
Tahakkuk 2 (Göynücek): (295223, 10)


Excel dosyasındaki 5 sayfa başarıyla yüklendi. Elde edilen boyutlar, case study'de
verilen referans değerlerle birebir eşleşiyor

In [5]:
#Sayfaların tek tek ınfo describe head bilgilerini alma

def sayfa_incele(df, isim):
    print(isim)
    print("------------------------------------")

    print("\n     INFO   \n")
    df.info()

    print("\n     DESCRIBE   \n")
    print(df.describe())

    print("\n     HEAD    \n")
    print(df.head())

In [6]:
sayfa_incele(df_tahsilat, "Tahsilat")

Tahsilat
------------------------------------

     INFO   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 636993 entries, 0 to 636992
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Şube                   636993 non-null  object        
 1   Kasa                   636993 non-null  object        
 2   İlçe                   636993 non-null  object        
 3   Söz.hsp.(bağımsız)     636993 non-null  int64         
 4   Tahsilat Tarihi        636993 non-null  datetime64[ns]
 5   Nakit Tahsilat         523 non-null     float64       
 6   Mahsuben Tahsilat      7542 non-null    float64       
 7   Kredi Kartı Tahsilatı  0 non-null       float64       
 8   Banka Tahsilatı        628933 non-null  float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(3)
memory usage: 43.7+ MB

     DESCRIBE   

       Söz.hsp.(bağımsız)                Tahsilat Tarihi  Nakit Tahs

* Sütunlarımız 9 adet doğru şekilde geldi.
* Veriye baktığımızda Taşova adında farklı bir ilçe daha görüyoruz yani veri sadece 3 ilçeden (Hamamözü, Gümüşhacıköy, Göynücek) değil daha fazla ilçeden oluşuyor.
* Ayrıca veriye bakınca her fatura için sadece 1 tahsilat tipi dolu diğerleri NaN olarak girilmiş. Burada veri kaybı var gibi düşünmeyiz bence.Veri var.


In [7]:
df_tahsilat['İlçe'].value_counts()

,count
İlçe,
TAŞOVA,289077
GÜMÜŞHACIKÖY,252818
GÖYNÜCEK,82519
HAMAMÖZÜ,12579


Gördüğümüz gibi aslında 4 farklı ilçemiz var. Taşova kayıtların büyük bir kısmını oluşturuyor.Ama case studyde onu ele almıyoruz.

In [29]:
nakit_sayisi = df_tahsilat['Nakit Tahsilat'].notna().sum()
mahsuben_sayisi = df_tahsilat['Mahsuben Tahsilat'].notna().sum()
kredi_karti_sayisi = df_tahsilat['Kredi Kartı Tahsilatı'].notna().sum()
banka_sayisi = df_tahsilat['Banka Tahsilatı'].notna().sum()

print("Nakit Tahsilat:", nakit_sayisi)
print("Mahsuben Tahsilat:", mahsuben_sayisi)
print("Kredi Kartı Tahsilatı:", kredi_karti_sayisi)
print("Banka Tahsilatı:", banka_sayisi)

Nakit Tahsilat: 523
Mahsuben Tahsilat: 7542
Kredi Kartı Tahsilatı: 0
Banka Tahsilatı: 628933


Tahsilat sayfasındaki 4 ödeme kanalının (Nakit, Mahsuben, Kredi Kartı, Banka) kayıt sayılarını
kontrol ettik. Sonuçlar case study'nin referans değerleriyle birebir örtüşüyor: Banka Tahsilatı en çok kullanılan kanal, onu Mahsuben Tahsilat
ve Nakit Tahsilat izliyor. Kredi Kartı Tahsilatı hiç kullanılmamış.

Ancak "en çok kullanılan kanal" ile "en büyük tutarların geçtiği kanal" aynı şey değil bu tahsilatlarda neye göre nakit neye göre bankadan aktarılıyor önemli.

In [33]:
kanal_karsilastirma = df_tahsilat[['Nakit Tahsilat', 'Mahsuben Tahsilat', 'Banka Tahsilatı']].agg(['mean', 'median','min','max'])
kanal_karsilastirma

,Nakit Tahsilat,Mahsuben Tahsilat,Banka Tahsilatı
mean,694.966635,6180.182282,372.629109
median,524.670000,290.410000,208.000000
min,7.450000,-34508.950000,0.010000
max,11373.740000,399526.780000,606473.800000


Ortalamaya bakılırsa Mahsuben Tahsilat en yüksek tutarlı kanal gibi görünüyor, ama medyanı diğer iki kanaldan bile düşük yani ortalamayı yukarı çeken, birkaç istisnai büyük işlem olabilir.

Normal işlem büyüklüğüne bakıldığında en yüksek medyan Nakit
Tahsilat'ta.
Banka Tahsilatı hem en çok kullanılan hem de genelde en küçük/rutin
tutarlı kanal — muhtemelen standart aylık fatura ödemelerini temsil ediyor.

**NOT:** Bir Tahsilat kaydı, bir müşterinin tek seferde yaptığı ödemeyi temsil
ediyor, ama müşteri o seferde birden fazla fatura ödemiş olabilir. Yani buradaki tutarlar tek
bir faturanın değil, o işlemde ödenen faturaların toplamını gösteriyor olabilir. Bu yüzden
"büyük tutar = pahalı fatura" sonucuna doğrudan varamayız. Bu ilişkiyi netleştirmek için
Tahsilat kayıtlarını fatura bazlı Tahsilat 1 verisiyle eşleştirmek gerekir.

In [8]:
sayfa_incele(df_tahsilat_1, "Tahsilat 1")

Tahsilat 1
------------------------------------

     INFO   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 917632 entries, 0 to 917631
Data columns (total 22 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Mali yıl/dönem                        917632 non-null  object 
 1   İl                                    917632 non-null  object 
 2   İlçe                                  917632 non-null  object 
 3   Söz.hsp.(bağımsız)                    917632 non-null  int64  
 4   Hesap Sınıfı                          917632 non-null  object 
 5   Tahakkuk Tutar                        917632 non-null  float64
 6   Son Ödeme Tarihinden Önceki Tahsilat  623908 non-null  float64
 7   Son Ödeme Tarihindeki Tahsilat        328193 non-null  float64
 8   Son Ödeme (1)                         20902 non-null   float64
 9   Son Ödeme (2)                         21664 non-null   float64
 10  Son Ö

* Bu veride de yine NaN değerler var ama bu veri kaybından dolayı değil — 16 farklı "son ödemezamanlaması" sütunundan bir kayıtta genelde sadece biri dolu oluyor, geri kalanı yapısal olarak boşkalıyor. Ama bazı kayıtlarda birden fazla sütun aynı anda dolu olabiliyor (örneğin head()çıktısındaki 0. satırda hem "Son Ödeme Tarihinden Önceki" hem "Son Ödeme (90-120)" dolu) bu da  case study de bahsedilen kısmi ödeme yapıldığını gösteriyor, bunu aşağıda ayrıca inceledim.
* Yine aynı şekilde bu veride de 4 tane ilçe var (Taşova dahil),


In [9]:
df_tahsilat_1['İlçe'].value_counts()

,count
İlçe,
TAŞOVA,435394
GÜMÜŞHACIKÖY,311264
GÖYNÜCEK,120832
HAMAMÖZÜ,50142


In [34]:
# Zamanında/erken ödeme
zamaninda_mi = df_tahsilat_1['Son Ödeme Tarihinden Önceki Tahsilat'].notna() | df_tahsilat_1['Son Ödeme Tarihindeki Tahsilat'].notna()
zamaninda_sayisi = zamaninda_mi.sum()

# Geç ödeme (16 sütundan sonraki 14 tanesi)
gec_sutunlari = ['Son Ödeme (1)', 'Son Ödeme (2)', 'Son Ödeme (3)', 'Son Ödeme (4)', 'Son Ödeme (5)',
                  'Son Ödeme (6-10)', 'Son Ödeme (10-20)', 'Son Ödeme (20-30)', 'Son Ödeme (30-60)',
                  'Son Ödeme (60-90)', 'Son Ödeme (90-120)', 'Son Ödeme (120-150)', 'Son Ödeme (150-180)',
                  'Son Ödeme (180+)']
gec_mi = df_tahsilat_1[gec_sutunlari].notna().any(axis=1)
gec_sayisi = gec_mi.sum()

toplam = len(df_tahsilat_1)
print("Zamanında/erken:", zamaninda_sayisi, "- %", round(zamaninda_sayisi/toplam*100, 2))
print("Geç:", gec_sayisi, "- %", round(gec_sayisi/toplam*100, 2))

# Kısmi ödeme: hem zamanında hem geç sütunu aynı anda dolu mu?
kismi_odeme_sayisi = (zamaninda_mi & gec_mi).sum()
print("Kısmi ödeme (hem zamanında hem geç):", kismi_odeme_sayisi, "- %", round(kismi_odeme_sayisi/toplam*100, 2))

# İlçeye göre zamanında/geç kayıt sayıları
df_tahsilat_1['zamaninda_mi'] = zamaninda_mi
df_tahsilat_1['gec_mi'] = gec_mi

ilce_ozet = pd.DataFrame({
    'zamaninda_sayisi': df_tahsilat_1.groupby('İlçe')['zamaninda_mi'].sum(),
    'gec_sayisi': df_tahsilat_1.groupby('İlçe')['gec_mi'].sum(),
    'toplam_kayit': df_tahsilat_1.groupby('İlçe').size()
})
ilce_ozet['zamaninda_yuzde'] = ilce_ozet['zamaninda_sayisi'] / ilce_ozet['toplam_kayit'] * 100
ilce_ozet['gec_yuzde'] = ilce_ozet['gec_sayisi'] / ilce_ozet['toplam_kayit'] * 100
print(ilce_ozet)

Zamanında/erken: 789588 - % 86.05
Geç: 249487 - % 27.19
Kısmi ödeme (hem zamanında hem geç): 123277 - % 13.43
              zamaninda_sayisi  gec_sayisi  toplam_kayit  zamaninda_yuzde  \
İlçe                                                                        
GÖYNÜCEK                101951       37181        120832        84.374172   
GÜMÜŞHACIKÖY            271526       77126        311264        87.233345   
HAMAMÖZÜ                 42694       14378         50142        85.146185   
TAŞOVA                  373417      120802        435394        85.765307   

              gec_yuzde  
İlçe                     
GÖYNÜCEK      30.770822  
GÜMÜŞHACIKÖY  24.778323  
HAMAMÖZÜ      28.674564  
TAŞOVA        27.745444  


Bu blokta dört şeyi kontrol etmek istedim:

1. **Zamanında/erken ödeme oranı** — case study'nin verdiği referans değerle (%86) karşılaştırıp verimin doğru olduğunu teyit etmek için hesapladım.
2. **Geç ödeme oranı** — aynı şekilde referans değerle (%27.2)karşılaştırmak için hesapladım.
3. **Kısmi ödeme kontrolü** — yukarıda fark ettiğim gibi, bazı kayıtlarda hem "zamanında"hem "geç" sütunu aynı anda dolu olabiliyor. Bunun tek bir istisna mı yoksa yaygın bir durummu olduğunu sayısal olarak görmek istedim, bu yüzden iki koşulu (zamaninda_mi ve gec_mi) ile birleştirip kaç kayıtta ikisinin birden geçerli olduğuna baktım.
4. **İlçeye göre ayrım** — daha önce ilçeler arasında tüketim davranışında farklar bulmuştuk,ödeme davranışında da benzer bir örüntü olup olmadığını merak ettim, bu yüzden zamanında/geç ödeme sayılarını ilçe bazında da hesapladım.

In [10]:
sayfa_incele(df_tahakkuk, "Tahakkuk (Hamamözü)")

Tahakkuk (Hamamözü)
------------------------------------

     INFO   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124818 entries, 0 to 124817
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   il                 124818 non-null  object 
 1   ilce               124818 non-null  object 
 2   sozlesme_hesap_no  124818 non-null  int64  
 3   mali_yil_donem     124818 non-null  object 
 4   fatura_tarihi      124818 non-null  object 
 5   kayit_tarihi       124818 non-null  object 
 6   vade_tarihi        124818 non-null  object 
 7   hesap_sinifi       124818 non-null  object 
 8   Hesap Sınıfı       124818 non-null  object 
 9   kwh                124818 non-null  float64
dtypes: float64(1), int64(1), object(8)
memory usage: 9.5+ MB

     DESCRIBE   

       sozlesme_hesap_no            kwh
count       1.248180e+05  124818.000000
mean        5.044916e+09      70.874619
std         2.874544e+09    

In [11]:
df_tahakkuk['ilce'].value_counts()

,count
ilce,
HAMAMÖZÜ,124818


* Kayıt sayısı (124,818) case study ile örtüşüyor, veri doğru yüklenmiş.
* head kısmında 2 tane hesap sınıfı verisi var, biri kod (hesap_sinifi) biri isim (Hesap Sınıfı) olarak.
* Tarih sütunları (mali_yil_donem, fatura_tarihi, kayit_tarihi, vade_tarihi) object olarak görünüyor, sanırım daha tarih olarak tanımlanmadılar.
* Ortalama ve medyan değerler case study ile uyuşuyor.
* kwh min değerinde negatif değerler var.
* kwh max değeri ortalamaya göre çok yüksek, outlier olabilir.

In [12]:
sayfa_incele(df_tahakkuk_1, "Tahakkuk 1 (Gümüşhacıköy)")

Tahakkuk 1 (Gümüşhacıköy)
------------------------------------

     INFO   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765657 entries, 0 to 765656
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   il                 765657 non-null  object 
 1   ilce               765657 non-null  object 
 2   sozlesme_hesap_no  765657 non-null  int64  
 3   mali_yil_donem     765657 non-null  object 
 4   fatura_tarihi      765657 non-null  object 
 5   kayit_tarihi       765657 non-null  object 
 6   vade_tarihi        765657 non-null  object 
 7   hesap_sinifi       765657 non-null  object 
 8   Hesap Sınıfı       765657 non-null  object 
 9   kwh                765657 non-null  float64
dtypes: float64(1), int64(1), object(8)
memory usage: 58.4+ MB

     DESCRIBE   

       sozlesme_hesap_no            kwh
count       7.656570e+05  765657.000000
mean        5.019916e+09      97.336632
std         2.881724e

In [13]:
df_tahakkuk_1['ilce'].value_counts()

,count
ilce,
GÜMÜŞHACIKÖY,765657


* Kayıt sayısı case study ile örtüşüyor, veri doğru yüklenmiş.
* head kısmında yine 2 tane hesap sınıfı verisi var (hesap_sinifi kod, Hesap Sınıfı isim).
* Tarih sütunları yine object, tarih olarak tanımlanmamış.
* Ortalama ve medyan değerler case study ile uyuşuyor.
* kwh min (-25,370.64) ve max (153,575.73) değerleri, tüm veri setinin (3 ilçe birleşik) genel min/max değerleriyle aynı — yani en uç değerler bu sayfadan geliyor.
* std değeri Hamamözü'nden çok daha yüksek, bu ilçede tüketim daha değişken.

In [14]:
sayfa_incele(df_tahakkuk_2, "Tahakkuk 2 (Göynücek)")

Tahakkuk 2 (Göynücek)
------------------------------------

     INFO   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295223 entries, 0 to 295222
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   il                 295223 non-null  object 
 1   ilce               295223 non-null  object 
 2   sozlesme_hesap_no  295223 non-null  int64  
 3   mali_yil_donem     295223 non-null  object 
 4   fatura_tarihi      295223 non-null  object 
 5   kayit_tarihi       295223 non-null  object 
 6   vade_tarihi        295223 non-null  object 
 7   hesap_sinifi       295223 non-null  object 
 8   Hesap Sınıfı       295223 non-null  object 
 9   kwh                295223 non-null  float64
dtypes: float64(1), int64(1), object(8)
memory usage: 22.5+ MB

     DESCRIBE   

       sozlesme_hesap_no            kwh
count       2.952230e+05  295223.000000
mean        4.950566e+09      89.669891
std         2.887724e+09 

In [15]:
df_tahakkuk_2['ilce'].value_counts()

,count
ilce,
GÖYNÜCEK,295223


* Kayıt sayısı case study ile örtüşüyor, veri doğru yüklenmiş.
* head kısmında yine 2 tane hesap sınıfı verisi var (hesap_sinifi kod, Hesap Sınıfı isim).
* Tarih sütunları yine object, tarih olarak tanımlanmamış.
* Ortalama ve medyan değerler case study ile uyuşuyor.
* kwh min (-4,208.64) ve max (105,687.69) yine uç değerler var ama Gümüşhacıköy kadar aşırı değil.
* head'de ilginç bir durum var: aynı müşterinin mali_yil_donem'i 2023 olduğu halde fatura_tarihi bazı satırlarda 2025'e sıçramış gecikmeli faturalama olabilir, bu satırlardaki kwh değerleri de neredeyse sıfır.

In [16]:
hamamozu_musteri = df_tahakkuk['sozlesme_hesap_no'].nunique()
print("Hamamözü benzersiz müşteri sayısı:", hamamozu_musteri)
print("\n")

gumushacikoy_musteri = df_tahakkuk_1['sozlesme_hesap_no'].nunique()
print("Gümüşhacıköy benzersiz müşteri sayısı:", gumushacikoy_musteri)
print("\n")

goynucek_musteri = df_tahakkuk_2['sozlesme_hesap_no'].nunique()
print("Göynücek benzersiz müşteri sayısı:", goynucek_musteri)
print("\n")

toplam_musteri = hamamozu_musteri + gumushacikoy_musteri + goynucek_musteri
print("Toplam benzersiz müşteri sayısı:", toplam_musteri)


Hamamözü benzersiz müşteri sayısı: 2981


Gümüşhacıköy benzersiz müşteri sayısı: 18190


Göynücek benzersiz müşteri sayısı: 7128


Toplam benzersiz müşteri sayısı: 28299


* Her ilçe için benzersiz müşteri sayısı case study ile birebir örtüşüyor.
* Gümüşhacıköy'ün müşteri sayısı Hamamözü'nün yaklaşık 6 katı, bu da o ilçenin neden daha fazla toplam kayda ve daha uç kwh değerlerine sahip olabileceğini açıklıyor.

In [17]:
df_tahakkuk_tumu = pd.concat([df_tahakkuk, df_tahakkuk_1, df_tahakkuk_2])
print("Toplam tahakkuk kayıt sayısı:", df_tahakkuk_tumu.shape[0])

Toplam tahakkuk kayıt sayısı: 1185698


In [18]:
eksik_sayisi = df_tahakkuk_tumu['kwh'].isna().sum()
print("kwh sütununda eksik değer sayısı:", eksik_sayisi)

kwh sütununda eksik değer sayısı: 0


In [19]:
negatif_sayisi = (df_tahakkuk_tumu['kwh'] < 0).sum()
print("kwh sütununda negatif değer sayısı:", negatif_sayisi)

kwh sütununda negatif değer sayısı: 151


In [20]:
negatif_veriler = df_tahakkuk_tumu[df_tahakkuk_tumu['kwh'] < 0]
negatif_veriler.head(10)

,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı,kwh
23573,AMASYA,HAMAMÖZÜ,7904741733,2025-03-01,2025-04-14,2025-06-10,2025-04-22,M001,Mesken,-1242.99
23576,AMASYA,HAMAMÖZÜ,7904741733,2025-04-01,2025-04-14,2025-06-10,2025-04-22,M001,Mesken,-733.04
28413,AMASYA,HAMAMÖZÜ,469034964,2023-10-01,2023-10-11,2023-11-28,2023-10-23,M001,Mesken,-1.04
70626,AMASYA,HAMAMÖZÜ,7722066065,2023-10-01,2023-10-23,2023-11-28,2023-11-02,M001,Mesken,-1.34
55683,AMASYA,GÜMÜŞHACIKÖY,2945215788,2024-02-01,2024-04-15,2024-06-15,2024-03-25,M001,Mesken,-374.09
55687,AMASYA,GÜMÜŞHACIKÖY,2945215788,2024-03-01,2024-04-15,2024-06-15,2024-03-25,M001,Mesken,-294.25
90755,AMASYA,GÜMÜŞHACIKÖY,3066487951,2023-01-01,2024-06-13,2024-08-26,2023-01-30,M001,Mesken,-15.48
90756,AMASYA,GÜMÜŞHACIKÖY,3066487951,2023-01-01,2024-06-13,2024-08-26,2023-02-27,M001,Mesken,-9.52
90758,AMASYA,GÜMÜŞHACIKÖY,3066487951,2023-02-01,2024-06-13,2024-08-26,2023-02-27,M001,Mesken,-13.48
105797,AMASYA,GÜMÜŞHACIKÖY,6095948483,2023-01-01,2025-10-01,2025-10-08,2024-08-15,M001,Mesken,-226.19


In [21]:
negatif_veriler['kwh'].describe()

,kwh
count,151.000000
mean,-743.667020
std,2501.081659
min,-25370.640000
25%,-381.820000
50%,-204.300000
75%,-14.480000
max,-0.010000


In [22]:
negatif_veriler['Hesap Sınıfı'].value_counts()

,count
Hesap Sınıfı,
Mesken,106
Tarımsal Faaliyetler (Kooperatif),16
Ticari Faaliyet - Yazıhane,13
Tarımsal Faaliyetler (Şahıs),10
Şantiye ve Geçici Aboneler,2
Resmi Daire,2
Belediye,1
Süt Toplama Merkezi,1


In [23]:
Q1 = df_tahakkuk_tumu['kwh'].quantile(0.25)
Q3 = df_tahakkuk_tumu['kwh'].quantile(0.75)
IQR = Q3 - Q1

alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Alt sınır:", alt_sinir)
print("Üst sınır:", ust_sinir)

Q1: 18.01
Q3: 80.0
IQR: 61.989999999999995
Alt sınır: -74.97499999999998
Üst sınır: 172.98499999999999


In [24]:
outlier_sayisi = ((df_tahakkuk_tumu['kwh'] < alt_sinir) | (df_tahakkuk_tumu['kwh'] > ust_sinir)).sum()
print("Outlier sayısı:", outlier_sayisi)

Outlier sayısı: 48554


In [25]:
dusuk_outlier = (df_tahakkuk_tumu['kwh'] < alt_sinir).sum()
yuksek_outlier = (df_tahakkuk_tumu['kwh'] > ust_sinir).sum()

print("Alt sınırın altındaki (aşırı düşük/negatif) outlier sayısı:", dusuk_outlier)
print("Üst sınırın üstündeki (aşırı yüksek) outlier sayısı:", yuksek_outlier)

Alt sınırın altındaki (aşırı düşük/negatif) outlier sayısı: 104
Üst sınırın üstündeki (aşırı yüksek) outlier sayısı: 48450


In [26]:
yuksek_outlier_veriler = df_tahakkuk_tumu[df_tahakkuk_tumu['kwh'] > ust_sinir]
yuksek_outlier_veriler['Hesap Sınıfı'].value_counts()

,count
Hesap Sınıfı,
Mesken,20853
Ticari Faaliyet - Yazıhane,15809
1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,2260
Tarımsal Faaliyetler (Şahıs),1889
Köy İçme Suyu Temini ve Dağıtımı Tesisi,1315
Tarımsal Faaliyetler (Kooperatif),984
İbadethane Isıtma/Soğutma/Lojman,924
Belediye,725
Resmi Daire,634


In [35]:

hesap_sinifi_istatistik = df_tahakkuk_tumu.groupby('Hesap Sınıfı')['kwh'].agg(['mean', 'median', 'std','min', 'max'])
hesap_sinifi_istatistik

,mean,median,std,min,max
Hesap Sınıfı,,,,,
1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,688.441598,23.860,3911.912974,0.00,76153.92
Aritma Tesisleri,16594.174857,16186.910,11656.705924,218.78,34510.46
Balıkçılık ve Su Ürünleri Yetiştiriciliğ,85.358306,79.375,42.120066,20.81,330.98
Belediye,600.581235,78.270,3350.978976,-25370.64,50741.28
Belediye Park Bahçe Aydınlatma,31.951003,18.390,38.472449,0.00,409.41
"Bina Ort Kul (Asn,Hidr,Kapıcı Dai vb.)",29.965663,15.760,41.331395,0.00,466.80
Büyükbaş-Küçükbaş Hayvancılık,116.848021,24.110,408.549664,0.00,4100.15
Cemevleri,10.713755,5.625,16.453958,0.00,236.49
Diyanet Kuran Kursu,167.135000,22.115,526.354274,0.00,2628.41


* kwh'de kaç eksik ve negatif değer olduğunu bulmam gerekiyordu (görev 4), ama sadece sayıyı
  bulup geçmek yerine bu 151 negatif değerin ne anlama geldiğini merak ettim, bu yüzden
  `head()`, `describe()` ve `Hesap Sınıfı` dağılımına baktım.
* Aşırı uç değerleri (outlier) tespit etmek için IQR yöntemini kullandım — sabit bir eşik
  koymak yerine verinin kendi dağılımına göre "normal" aralığı belirlemek istedim.
* Outlier sayısını alt/üst sınır olarak ayırdım, çünkü negatif değerlerin (151) toplam
  outlier sayısını (48,554) açıklamaya yetmediğini fark ettim, kaynağını merak ettim.
* Üst sınırın üstündeki outlier'ların hesap sınıfı dağılımına baktım, çünkü bunun gerçek bir
  veri hatası mı yoksa farklı müşteri tipinden (sanayi/ticari gibi) mi kaynaklandığını
  ayırt etmek istedim.

### SONUÇ OLARAK
* 151 negatif kwh değerinin çoğu küçük/orta büyüklükte , ama bazı müşterilerdeaynı fatura tarihiyle birden fazla döneme yayılmış negatif kayıtlar var — bu, gerçek negatif tüketim değil, geriye dönük fatura ödemesi olabilir mi?
* Negatif değerlerde tarımsal hesaplar (Kooperatif+Şahıs) genel paydaki oranlarına göre fazla.

* IQR yöntemiyle 48,554 outlier bulundu,bunun %99.8'i  üst sınırdan geliyor,sadece 104'ü alt sınırdan.

* Üst outlier'larda Mesken hâlâ en kalabalık grup ama genel paydaki payına göre düşük temsil ediliyor; Ticari Faaliyet-Yazıhane orantısız fazla .
* Yani outlier'ların büyük kısmı veri hatası değil, farklı müşteri tipinden (ticari, kamu altyapısı gibi) kaynaklanıyor gibi gözüküyor.